In [3]:
import duckdb

con = duckdb.connect()

In [4]:
path = "s3://mateom/graal/embeddings/NAF2025/2026-03-16_gemma_synth.parquet"
n_samples = 1000

In [ ]:
con.execute(f"SELECT setseed({0.5});")
query = f"""
SELECT code, label, embedding
FROM read_parquet('{path}')
USING SAMPLE {n_samples} ROWS
ORDER BY (code, label)
"""

df = con.execute(query).df()
df

,code,label,embedding
0,01.11Y,Culture de céréales diversifiées,"[0.030173147097229958, 0.029588395729660988, 0..."
1,01.11Y,Exploitation d'une activité agricole axée sur ...,"[0.038778554648160934, 0.023314714431762695, 0..."
2,01.11Y,"Intervention sur culture de légumineuse, contr...","[0.040246400982141495, 0.001985842129215598, 0..."
3,01.11Y,Production de graines de niger,"[0.018320737406611443, 0.025719495490193367, 0..."
4,01.11Y,"Production de graines de soja, arachides et ri...","[0.0009564123465679586, 0.037485431879758835, ..."
...,...,...,...
995,96.99H,Activités de bien-être alternatives,"[0.024193717166781425, 0.021492475643754005, 0..."
996,96.99H,"Préposé au parcage, organisation de stationnem...","[0.009673898108303547, 0.013768983073532581, 0..."
997,98.10Y,Ménages produisant pour leur propre compte.,"[0.00983060896396637, 0.03581150248646736, -0...."
998,98.20Y,Cuisine et soins assurés au sein du foyer.,"[0.010468038730323315, 0.023656584322452545, 0..."


In [5]:
con.execute(f"SELECT setseed({0.5});")
query = f"""
SELECT COUNT(DISTINCT(code))
FROM read_parquet('{path}')
"""

df = con.execute(query).df()
df

,count(DISTINCT code)
0,747


In [7]:
df.loc[0,"embedding"]

array([ 0.03017315,  0.0295884 ,  0.00701701, ..., -0.00229515,
       -0.01122722,  0.00882974], shape=(4096,))

In [1]:
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv

load_dotenv(override=True)

QDRANT_URL = "http://qdrant:6333"
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]  # ou None si local

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=30
)

/tmp/ipykernel_66813/670185661.py:10: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [ ]:
from qdrant_client.models import VectorParams, Distance, PointStruct

# CONFIG
COLLECTION_NAME = "test_collection"
VECTOR_SIZE = 4096

# Reset collection
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
)

# Insert 1 point
point = PointStruct(
    id=1,
    vector=[0.1] * VECTOR_SIZE,
    payload={"test": "ok"}
)

client.upsert(
    collection_name=COLLECTION_NAME,
    points=[point]
)

print("✅ Insert OK")

# Insert batch (stress test léger)
points = [
    PointStruct(id=i, vector=[0.1] * VECTOR_SIZE, payload={"i": i})
    for i in range(100)
]

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print("✅ Batch insert OK")

/tmp/ipykernel_2700/1007922045.py:13: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


✅ Insert OK
✅ Batch insert OK


In [13]:
code_hits = client.facet(
    collection_name="original_cleaned",
    key="code",
    exact=True,
    limit=1000
).hits

code_list = {hit.value: hit.count for hit in code_hits}

In [17]:
code_list.keys()

dict_keys(['68.20H', '70.20Y', '68.20G', '96.22Y', '62.10Y', '43.21G', '95.31G', '62.20G', '85.59H', '73.30Y', '82.10Y', '66.30Y', '71.12Y', '85.59G', '69.10Y', '55.20Y', '49.41G', '70.10Y', '43.22G', '96.23Y', '46.19H', '43.32G', '81.30Y', '56.21Y', '47.12H', '73.11Y', '43.31Y', '86.21Y', '86.22Y', '94.99Y', '58.13Y', '32.13Y', '96.99H', '53.20H', '43.22H', '85.52Y', '66.22Y', '74.30Y', '43.33Y', '47.71Y', '68.11Y', '46.90Y', '86.23Y', '43.41H', '56.30Y', '77.11Y', '43.12G', '88.10G', '01.11Y', '69.20Y', '71.11Y', '01.21Y', '80.01Y', '81.22Y', '95.10Y', '10.71H', '01.61Y', '59.20Y', '85.69Y', '88.99H', '01.48J', '55.10Y', '01.13Y', '74.12Y', '58.29Y', '68.32G', '85.53Y', '33.20Y', '71.20H', '43.41G', '74.99Y', '01.50Y', '47.76Y', '46.42Y', '10.71J', '58.12Y', '68.32H', '59.12Y', '86.92Y', '59.11G', '77.22Y', '85.40Y', '01.48G', '43.23Y', '75.00Y', '47.40Y', '47.75Y', '47.22Y', '23.41Y', '73.20Y', '82.20Y', '81.10Y', '10.72Y', '01.45Y', '46.83G', '81.23G', '81.23H', '01.42Y', '01.43Y',

In [3]:
import numpy as np

In [22]:
collection_name = "original_cleaned"
code = "68.20H"
rng = np.random.default_rng(1)
size = 1000

records, _ = client.scroll(
    collection_name=collection_name,
    scroll_filter={
        "must": [
            {"key": "code", "match": {"value": code}}
        ]
    },
    with_payload=False,
    with_vectors=False,
    limit=100000
)

ids = [record.id for record in records]

random_ids = rng.choice(ids, size=size, replace=True)

random_points = client.retrieve(
    collection_name=collection_name,
    ids=random_ids,
    with_payload=True,
    with_vectors=True,
)

In [23]:
id_to_points = {}

for p in random_points:
    id_to_points[p.id] = p

In [24]:
final_points = [id_to_points[id] for id in random_ids]

In [26]:
payloads = [point.payload for point in final_points]
vectors = [point.vector for point in final_points]

In [66]:
y = np.append(np.zeros(len(payloads) - 500), np.ones(500))
y.astype(np.float32)
for payload, is_synth in zip(payloads, y):
        payload["is_synth"] = int(is_synth)

In [41]:
selected = rng.choice(100000, size=int(0.8*100000), replace=False)
selected.sort()

In [42]:
not_selected = np.delete(np.arange(100000), selected)

In [18]:
len(final_points)

10000

In [2]:
import numpy as np

In [10]:
rng = np.random.default_rng(1)
indices = np.arange(100000)
indices_train = rng.choice(indices, size=int(0.8*100000), replace=False)
indices_test = np.delete(indices, indices_train)
rng.shuffle(indices_test)

In [11]:
payloads = list(np.linspace(127, 129182, 100000))

In [12]:
payloads_train = [payloads[i] for i in indices_train]
payloads_test = [payloads[i] for i in indices_test]

In [13]:
payloads_test[:10]

[np.float64(65219.121271212716),
 np.float64(8795.711037110372),
 np.float64(77753.06821068212),
 np.float64(105523.40081400814),
 np.float64(74827.36210362104),
 np.float64(73634.8819788198),
 np.float64(72903.13281132812),
 np.float64(101589.76507765078),
 np.float64(38073.421114211145),
 np.float64(3521.1804418044185)]